# Siamese Defect Detector — run locally

Runs this repo's pipeline **in-process, against the files already on disk** — no `git clone`, no Google Drive mount, no file upload. Meant for VS Code's built-in Jupyter/notebook support, running against a local Python environment that already has `requirements.txt` installed (e.g. the `omnisee` conda env used earlier in this project).

**Before running:** in VS Code, use the kernel picker (top right of the notebook) to select the Python environment that has this project's dependencies installed (`torch`, `torchvision`, `albumentations>=2.0`, `opencv-python-headless`, ...). If you haven't installed them yet, run the optional `pip install` cell in section 2.

See `README.md` and `DESIGN.md` for what this model does; this notebook just runs it.

## 0. Locate the repo root

Works whether this notebook is launched with the repo root as the working directory, or from inside `notebooks/`.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd if (cwd / "configs" / "config.yaml").exists() else cwd.parent
assert (repo_root / "configs" / "config.yaml").exists(), (
    f"Could not find configs/config.yaml from {cwd} -- open this notebook from the repo "
    f"root or its notebooks/ subfolder."
)
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo root:", repo_root)

## 1. Load the config

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg

## 2. (Optional) Install dependencies

Skip this if your selected kernel's environment already has `requirements.txt` installed.

In [ ]:
# %pip install -r requirements.txt

## 3. (Optional) Prepare data from a COCO export

Skip this if `data/prepared/` already exists (it does in this checkout, built from
`data/train/_annotations.coco.json` + `data/valid/_annotations.coco.json` + `data/reference/`).
Re-run this if you add a new COCO export folder -- see
`scripts/convert_coco_to_siamese.py --help` for all options (per-image reference maps,
category renaming, etc.).

In [ ]:
# from scripts.convert_coco_to_siamese import main as convert_main
# import sys as _sys
# _sys.argv = [
#     "convert_coco_to_siamese",
#     "--coco-json", "data/train/_annotations.coco.json",
#     "--reference-image", "ChatGPT-original.png",
#     "--file-name-prefix", "train/",
#     "--output", "data/prepared/annotations/train.json",
#     "--images-dir", "data", "--reference-dir", "data/reference",
#     "--copy-to", "data/prepared",
# ]
# convert_main()

## 4. Sanity-check the dataset

Verifies every `file_name`/`reference_image` in `train_split`/`val_split` resolves on disk
and every category maps into `cfg["data"]["defect_classes"]`, before spending time training
on a broken path.

In [ ]:
import json
from collections import defaultdict

class_to_id = {name: i for i, name in enumerate(cfg["data"]["defect_classes"])}


def check_split(name: str):
    path = cfg["data"][f"{name}_split"]
    coco = json.load(open(path))
    cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)

    missing, n_boxes = 0, 0
    for img in coco["images"]:
        ip = os.path.join(cfg["data"]["images_dir"], img["file_name"])
        rp = os.path.join(cfg["data"]["reference_dir"], img["reference_image"])
        if not os.path.isfile(ip) or not os.path.isfile(rp):
            missing += 1
            print("MISSING", img["id"], ip, rp)
        for ann in anns_by_image.get(img["id"], []):
            class_to_id[cat_id_to_name[ann["category_id"]]]  # KeyError if unmapped
            n_boxes += 1
    print(f"{name}: images={len(coco['images'])} boxes={n_boxes} missing_paths={missing}")


check_split("train")
check_split("val")

## 5. Train

Calls `training.train.train(cfg)` directly (the same function `python -m training.train`
uses), so logs/progress bars stream into this cell's output. Checkpoints land in
`cfg["experiment"]["output_dir"]` (`best.pt`).

In [ ]:
from training.train import train as run_training

run_training(cfg)

## 6. Run inference on one validation pair

In [ ]:
import cv2
import torch

from inference.predict import load_model, predict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_path = os.path.join(cfg["experiment"]["output_dir"], "best.pt")
model = load_model(cfg, weights_path, device)

val_coco = json.load(open(cfg["data"]["val_split"]))
sample = val_coco["images"][0]
reference_path = os.path.join(cfg["data"]["reference_dir"], sample["reference_image"])
input_path = os.path.join(cfg["data"]["images_dir"], sample["file_name"])

reference_bgr = cv2.imread(reference_path)
input_bgr = cv2.imread(input_path)

result = predict(model, reference_bgr, input_bgr, cfg, device)
result

## 7. Visualize the detections

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt

size_hw = tuple(cfg["data"]["image_size"])  # (H, W)
display_bgr = cv2.resize(input_bgr, (size_hw[1], size_hw[0]))
display_rgb = cv2.cvtColor(display_bgr, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(display_rgb)
for inst in result["instances"]:
    x1, y1, x2, y2 = inst["bounding_box"]
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    linewidth=2, edgecolor="red", facecolor="none"))
    ax.text(x1, max(y1 - 5, 0), f"{inst['class']} {inst['confidence']:.2f}",
            color="red", fontsize=10, backgroundcolor="white")
ax.set_title(f"embedding_distance={result['embedding_distance']:.3f}  "
             f"unknown_defect_suspected={result['unknown_defect_suspected']}")
ax.axis("off")
plt.show()

## 8. (Optional) Export to ONNX / OpenVINO

Requires `onnx`, `onnxruntime`, `openvino`, `openvino-dev` (see `requirements.txt`).

In [ ]:
# from export.export_openvino import export_onnx, convert_to_openvino
#
# onnx_path = "export/artifacts/siamese_defect.onnx"
# os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
# export_onnx(cfg, weights_path, onnx_path)
# convert_to_openvino(onnx_path, "export/artifacts/siamese_defect", precision="FP16")